In [ ]:
# %% Cell 1: Setup & Dependencies
# ============================================================================
# CELL 1: ENVIRONMENT SETUP
# ============================================================================
# IMPORTANT: Kaggle has torch pre-installed. Installing insightface normally
# pulls in albumentations which conflicts with Kaggle's torch, causing:
#   SystemError: >() method: bad call flags
#
# Fix: Install insightface with --no-deps, then install only needed deps.
# Then monkey-patch sys.modules to skip the problematic mask_renderer import.
# ============================================================================

import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

def install_nodeps(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", package])

# Kaggle already has: torch, torchvision, onnxruntime-gpu, opencv
# DO NOT reinstall them — it breaks everything.

# Install insightface WITHOUT pulling albumentations/torch deps
install_nodeps("insightface")

# Install only the insightface deps that DON'T conflict with Kaggle
install("prettytable")
install("easydict")
install("Cython")

# Other dependencies (safe to install)
install("onnxruntime-gpu")     # Required by insightface (not always pre-installed)
install("faiss-cpu")
install("supervision")
install("lap")

# ViT Recognition Model (AdaFace ViT from HuggingFace)
install("transformers")
install("huggingface_hub")
install("safetensors")
install("timm")
install("omegaconf")
 
# GGA / InstantID (SDXL + ControlNet) — need >=0.27.0 for InstantID pipeline
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "diffusers>=0.27.0"])
install("accelerate")

print("✅ All dependencies installed!")

In [ ]:
# %% Cell 2: Imports & Configuration
# ============================================================================
# CELL 2: IMPORTS & CONFIGURATION
# ============================================================================
# We monkey-patch sys.modules to skip insightface.app.mask_renderer,
# which tries to import albumentations (which breaks Kaggle's torch).
# We don't need mask_renderer — it's for face mask overlay, not recognition.
# ============================================================================

import os
import sys
import types
import cv2
import numpy as np
import faiss
import json
import time
import warnings
from pathlib import Path
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict

# ── Monkey-patch: skip mask_renderer (it needs albumentations which breaks torch) ──
_fake_mask_renderer = types.ModuleType("insightface.app.mask_renderer")
sys.modules["insightface.app.mask_renderer"] = _fake_mask_renderer

import insightface
from insightface.app import FaceAnalysis
import supervision as sv

warnings.filterwarnings("ignore")

# ── Configuration ──────────────────────────────────────────────────────────
@dataclass
class Config:
    """All pipeline configuration in one place."""
    
    # === PATHS (Kaggle) ===
    # Update these if running locally vs Kaggle
    ENROLLMENT_DIR: str = "/kaggle/input/datasets/aadityas33011/vitdataset/vidData/solo-pics"   # 6 enrollment photos
    VIDEO_DIR: str = "/kaggle/input/datasets/aadityas33011/vitdataset/vidData/video"            # 3 test videos
    OUTPUT_DIR: str = "/kaggle/working/output"
    
    # === DETECTION (SCRFD) ===
    DET_MODEL: str = "buffalo_l"           # Includes SCRFD-10G detector
    DET_SIZE: Tuple[int, int] = (640, 640) # Detection input resolution
    DET_THRESH: float = 0.35               # Lowered: catch more faces (was 0.5)
    
    # === RECOGNITION ===
    EMBEDDING_DIM: int = 512               # ArcFace/LVFace output dimension
    
    # === MATCHING ===
    MATCH_THRESHOLD_BASE: float = 0.28     # Lowered: 0.4 was rejecting valid matches
    MATCH_THRESHOLD_LAMBDA: float = 0.10   # Reduced quality penalty
    UNKNOWN_LABEL: str = "Unknown"
    
    # === TRACKING ===
    TRACK_THRESH: float = 0.20             # Lower: keep more tracks alive
    TRACK_BUFFER: int = 60                 # Doubled: keep lost tracks longer
    MATCH_THRESH: float = 0.7              # Slightly relaxed IoU matching
    VOTE_WINDOW: int = 4                   # Smaller window for faster lock-in
    VOTE_MIN: int = 2                      # Lowered: 2/4 votes confirms identity
    
    # === GENERATIVE AUGMENTATION ===
    ENABLE_GGA: bool = True               # ENABLED for InstantID augmentation
    GGA_FALLBACK: bool = True              # Fall back to traditional aug if InstantID fails
    GGA_PROMPTS: List[str] = field(default_factory=lambda: [
        "a photorealistic portrait of a person in harsh bright sunlight, highly detailed",
        "a photorealistic portrait of a person looking slightly to the side, natural expression",
        "a photorealistic portrait of a person in dim cinematic lighting, sharp focus",
        "a photorealistic portrait of a person looking slightly down, indoor office lighting",
    ])
    
    # === VIDEO PROCESSING ===
    PROCESS_EVERY_N_FRAMES: int = 2        # Process more frames (was 3)
    RECOGNIZE_EVERY_N_FRAMES: int = 2      # Recognize EVERY processed frame (was 6)
    MAX_FRAMES: int = 900                  # Cap for long videos (30s @ 30fps)
    
    # === VISUALIZATION ===
    BBOX_COLOR: Tuple[int, int, int] = (0, 255, 0)
    UNKNOWN_COLOR: Tuple[int, int, int] = (0, 0, 255)
    FONT_SCALE: float = 0.6
    FONT_THICKNESS: int = 2

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

print(f"📁 Enrollment dir: {config.ENROLLMENT_DIR}")
print(f"🎥 Video dir:      {config.VIDEO_DIR}")
print(f"📂 Output dir:     {config.OUTPUT_DIR}")
print(f"✅ Configuration loaded!")

In [ ]:
# %% Cell 3: Face Detection Module (SCRFD via InsightFace)
# ============================================================================
# CELL 3: FACE DETECTION — SCRFD
# ============================================================================
# InsightFace's `buffalo_l` model pack includes SCRFD-10G for detection
# and a ResNet-100 ArcFace model for recognition. Both loaded here.
# ============================================================================

class FaceDetector:
    """
    SCRFD-based face detector via InsightFace.
    
    Why SCRFD over YOLO:
    - ~95% AP on WIDER FACE Hard (vs ~82% for YOLO variants)
    - Anchor-free design handles extreme aspect ratios in crowds
    - Outputs precise 5-point landmarks for proper alignment
    - Optimized compute redistribution for small faces (back-row students)
    """
    
    def __init__(self, config: Config):
        print("🔍 Loading SCRFD face detector...")
        self.app = FaceAnalysis(
            name=config.DET_MODEL,
            providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
        )
        self.app.prepare(ctx_id=0, det_size=config.DET_SIZE, det_thresh=config.DET_THRESH)
        self.config = config
        # Store the recognition model reference for direct embedding extraction
        # Note: self.app.models is a DICT (key=name, value=model object)
        self.rec_model = None
        for name, model in self.app.models.items():
            taskname = getattr(model, 'taskname', '')
            if taskname == 'recognition':
                self.rec_model = model
                print(f"   ✅ Found recognition model: {name} (taskname={taskname})")
                break
        if self.rec_model is None:
            # Last resort: look for any model with 112x112 input
            for name, model in self.app.models.items():
                input_size = getattr(model, 'input_size', None)
                if input_size and input_size == (112, 112):
                    self.rec_model = model
                    print(f"   ✅ Found recognition model by input size: {name}")
                    break
        print(f"   ✅ Detector ready (input size: {config.DET_SIZE})")
        print(f"   ✅ Recognition model extracted: {self.rec_model is not None}")
    
    def get_embedding_direct(self, aligned_face: np.ndarray) -> Optional[np.ndarray]:
        """
        Extract embedding DIRECTLY from an aligned 112×112 face chip,
        bypassing detection entirely. Critical for augmented variants.
        """
        if self.rec_model is None:
            return None
        try:
            # The insightface recognition model expects BGR, 112×112
            if aligned_face.shape[:2] != (112, 112):
                aligned_face = cv2.resize(aligned_face, (112, 112))
            # Use the model's get_feat or get method
            if hasattr(self.rec_model, 'get_feat'):
                # Preprocess: HWC BGR → CHW RGB normalized
                blob = cv2.dnn.blobFromImage(
                    aligned_face, 1.0/127.5, (112, 112), (127.5, 127.5, 127.5), swapRB=True
                )
                embedding = self.rec_model.session.run(
                    [self.rec_model.session.get_outputs()[0].name],
                    {self.rec_model.session.get_inputs()[0].name: blob}
                )[0][0]
                return embedding
            elif hasattr(self.rec_model, 'get'):
                from insightface.utils.face_align import norm_crop
                embedding = self.rec_model.get(aligned_face)
                return embedding
        except Exception as e:
            print(f"      ⚠️ Direct embedding failed: {e}")
        return None
    
    def detect(self, frame: np.ndarray) -> list:
        """
        Detect all faces in a frame.
        
        Returns:
            List of insightface.Face objects with:
            - .bbox: [x1, y1, x2, y2]
            - .kps: 5 facial landmarks (2D array, shape [5, 2])
            - .det_score: detection confidence
            - .embedding: 512-d vector (already computed by buffalo_l)
        """
        faces = self.app.get(frame)
        return faces
    
    def detect_for_enrollment(self, image: np.ndarray):
        """
        Detect the single dominant face in an enrollment photo.
        Returns the face with highest confidence, or None.
        """
        faces = self.detect(image)
        if not faces:
            return None
        # Return the face with highest detection score
        return max(faces, key=lambda f: f.det_score)
    
    def visualize(self, frame: np.ndarray, faces: list) -> np.ndarray:
        """Draw bounding boxes and landmarks on a frame."""
        vis = frame.copy()
        for face in faces:
            bbox = face.bbox.astype(int)
            cv2.rectangle(vis, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 255, 0), 2)
            if face.kps is not None:
                for kp in face.kps:
                    cv2.circle(vis, (int(kp[0]), int(kp[1])), 2, (0, 0, 255), -1)
        return vis

# Initialize detector
detector = FaceDetector(config)
print("✅ Face detector initialized!")

In [ ]:
# %% Cell 3B: Vision Transformer Recognition Model (AdaFace ViT)
# ============================================================================
# CELL 3B: VISION TRANSFORMER — AdaFace ViT-Base with KP-RPE
# ============================================================================
# The core ViT upgrade: Replaces CNN-based ArcFace (ResNet-50) with a
# Vision Transformer trained with AdaFace quality-adaptive margin loss.
#
# WHY ViT > CNN for classrooms:
#   - Self-attention dynamically weights face regions
#   - Occluded patches (hand, mask, beard) get LOW attention scores
#   - Periocular (eye) region gets HIGH attention scores
#   - Result: robust recognition despite partial occlusion
#
# Model: minchul/cvlface_adaface_vit_base_kprpe_webface12m
#   - ViT-Base backbone + KeyPoint Relative Position Encoding
#   - Trained on WebFace12M (12 million faces)
#   - Input: 112×112 RGB face + 5 keypoints (from SCRFD)
#   - Output: 512-d L2-normalized embedding
# ============================================================================

class ViTFaceRecognizer:
    """
    AdaFace ViT-Base face recognition model with KeyPoint Relative Position
    Encoding (KP-RPE).
    
    This is a genuine Vision Transformer — the image is split into patches
    (tokens), and multi-head self-attention computes relationships between
    ALL patch pairs. This gives the model a GLOBAL receptive field from
    layer 1, unlike CNNs which build local features bottom-up.
    
    KP-RPE advantage: Uses the 5-point facial landmarks as positional
    encoding bias. This means the model "knows" where the eyes, nose, and
    mouth should be — even when some are occluded.
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.model = None
        self.available = False
        self.device = "cpu"
        
        self._try_load_vit()
    
    def _try_load_vit(self):
        """Attempt to load AdaFace ViT from HuggingFace. Fails gracefully."""
        try:
            import torch
            from huggingface_hub import hf_hub_download
            
            print("🧠 Loading AdaFace ViT-Base (KP-RPE) from HuggingFace...")
            
            repo_id = "minchul/cvlface_adaface_vit_base_kprpe_webface12m"
            cache_path = os.path.expanduser("~/.cvlface_cache/adaface_vit")
            os.makedirs(cache_path, exist_ok=True)
            
            # Download required files
            print("   ⏳ Downloading model files...")
            required_files = ['config.json', 'wrapper.py', 'model.safetensors']
            
            # First download files.txt to get the full file list
            try:
                hf_hub_download(repo_id, 'files.txt', local_dir=cache_path, 
                               local_dir_use_symlinks=False)
                with open(os.path.join(cache_path, 'files.txt'), 'r') as f:
                    extra_files = [line.strip() for line in f.read().split('\n') if line.strip()]
                required_files = list(set(required_files + extra_files))
            except Exception:
                pass  # files.txt not available, use defaults
            
            for fname in required_files:
                full_path = os.path.join(cache_path, fname)
                if not os.path.exists(full_path):
                    try:
                        hf_hub_download(repo_id, fname, local_dir=cache_path,
                                       local_dir_use_symlinks=False)
                    except Exception as e:
                        print(f"      ⚠️ Could not download {fname}: {e}")
            
            # ─── Pre-load the compiled RPE CUDA extension ───
            print("   🔧 Pre-loading RPE CUDA extension...")
            
            site_packages_dirs = [
                os.path.expanduser("~/.local/lib/python3.12/site-packages"),
                os.path.expanduser("~/.local/lib/python3.11/site-packages"),
                os.path.expanduser("~/.local/lib/python3.10/site-packages"),
            ]
            for sp_dir in site_packages_dirs:
                if os.path.exists(sp_dir):
                    for f in os.listdir(sp_dir):
                        if f.startswith("rpe_index") and f.endswith(".egg"):
                            egg_path = os.path.join(sp_dir, f)
                            if egg_path not in sys.path:
                                sys.path.insert(0, egg_path)
                                print(f"      ✅ Added RPE egg: {egg_path}")
            
            try:
                import rpe_index_cpp
                print("      ✅ rpe_index_cpp available")
            except ImportError:
                print("      ⚠️ rpe_index_cpp not compiled yet")
            
            # ─── Monkey-patch sys.exit (RPE __init__.py calls it) ───
            _original_sys_exit = sys.exit
            sys.exit = lambda *a, **kw: None  # no-op
            
            # ─── Clear stale module cache ───
            stale_patterns = ['rpe_index', 'RPE', 'vit_kprpe', 'kprpe', 
                              'models.vit_kprpe', 'models.base',
                              'transformers_modules']
            mods_to_clear = [k for k in list(sys.modules.keys()) 
                            if any(p in k for p in stale_patterns)]
            for mod in mods_to_clear:
                del sys.modules[mod]
            if mods_to_clear:
                print(f"      🧹 Cleared {len(mods_to_clear)} stale modules")
            
            # ─── DIRECT LOAD (bypass AutoModel meta tensor issue) ───
            # AutoModel.from_pretrained creates model on "meta" device first,
            # but the ViT init code calls .item() which fails on meta tensors.
            # Fix: load the model directly via PyTorch.
            print("   📦 Loading model directly (bypassing AutoModel)...")
            
            cwd = os.getcwd()
            os.chdir(cache_path)
            if cache_path not in sys.path:
                sys.path.insert(0, cache_path)
            
            try:
                # Read the model config
                import json
                with open(os.path.join(cache_path, 'config.json'), 'r') as f:
                    config_json = json.load(f)
                
                # The config.json has a 'conf' field with model architecture params
                from omegaconf import OmegaConf
                model_conf = OmegaConf.create(config_json.get('conf', config_json))
                
                # Import the model factory directly
                from models import get_model
                self.model = get_model(model_conf)
                
                # Load pretrained weights
                weight_path = os.path.join(cache_path, 'pretrained_model', 'model.pt')
                if os.path.exists(weight_path):
                    print(f"      📥 Loading weights from model.pt...")
                    self.model.load_state_dict_from_path(weight_path)
                else:
                    # Try loading from safetensors
                    safetensors_path = os.path.join(cache_path, 'model.safetensors')
                    if os.path.exists(safetensors_path):
                        print(f"      📥 Loading weights from model.safetensors...")
                        from safetensors.torch import load_file
                        state_dict = load_file(safetensors_path)
                        # Remove 'model.' prefix if present (wrapper adds it)
                        cleaned = {}
                        for k, v in state_dict.items():
                            clean_key = k.replace('model.', '', 1) if k.startswith('model.') else k
                            cleaned[clean_key] = v
                        self.model.load_state_dict(cleaned, strict=False)
                    else:
                        raise FileNotFoundError("No weight file found (model.pt or model.safetensors)")
                
                print("      ✅ Weights loaded!")
                
            finally:
                sys.exit = _original_sys_exit
                os.chdir(cwd)
            
            # Move to GPU
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
            self.model = self.model.to(self.device)
            self.model.eval()
            
            self.available = True
            print(f"   ✅ AdaFace ViT loaded! (device: {self.device})")
            print(f"   📐 Architecture: ViT-Base + KeyPoint Relative Position Encoding")
            print(f"   📊 Training data: WebFace12M (12M faces)")
            
        except Exception as e:
            print(f"   ⚠️  AdaFace ViT failed to load: {e}")
            import traceback
            traceback.print_exc()
            print(f"   📸 Will use ArcFace ResNet-50 fallback for recognition.")
            self.available = False
    
    def get_embedding(self, face_chip: np.ndarray, keypoints: np.ndarray = None) -> Optional[np.ndarray]:
        """
        Extract 512-d embedding from an aligned face using the ViT model.
        
        Args:
            face_chip: Aligned 112×112 BGR face image
            keypoints: Optional 5×2 array of facial landmarks (for KP-RPE)
            
        Returns:
            512-d L2-normalized embedding, or None if model unavailable
        """
        if not self.available or self.model is None:
            return None
        
        try:
            import torch
            from torchvision.transforms import Compose, ToTensor, Normalize
            from PIL import Image
            
            # Convert BGR → RGB PIL Image, ensure 112×112
            face_rgb = cv2.cvtColor(face_chip, cv2.COLOR_BGR2RGB)
            if face_rgb.shape[:2] != (112, 112):
                face_rgb = cv2.resize(face_rgb, (112, 112))
            face_pil = Image.fromarray(face_rgb)
            
            # Standard normalization for face recognition models
            transform = Compose([
                ToTensor(),
                Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
            input_tensor = transform(face_pil).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                # ─── KEYPOINT SETUP ───────────────────────────────────────
                # The KP-RPE model expects keypoints in [0, 1] NORMALIZED
                # coordinates. The relative_keypoints.py builds its grid via
                # torch.linspace(0, 1, ...) and computes diff = grid - kps.
                # Passing pixel coords [38, 92] instead of [0.34, 0.82]
                # corrupts the entire attention mechanism.
                #
                # Since input faces are aligned to the standard 112×112
                # template, keypoints ARE at these canonical pixel positions.
                # We normalize them to [0, 1] by dividing by 112.
                # ──────────────────────────────────────────────────────────
                std_keypoints = np.array([
                    [38.2946 / 112.0, 51.6963 / 112.0],  # left eye
                    [73.5318 / 112.0, 51.5014 / 112.0],  # right eye
                    [56.0252 / 112.0, 71.7366 / 112.0],  # nose
                    [41.5493 / 112.0, 92.3655 / 112.0],  # left mouth
                    [70.7299 / 112.0, 92.2041 / 112.0],  # right mouth
                ], dtype=np.float32)
                
                try:
                    kps_tensor = torch.tensor(
                        std_keypoints, dtype=torch.float32
                    ).unsqueeze(0).to(self.device)
                    embedding = self.model(input_tensor, kps_tensor)
                except (TypeError, RuntimeError):
                    # Model might not accept keypoints in this format
                    embedding = self.model(input_tensor)
            
            # The model returns a single 512-d tensor directly
            if isinstance(embedding, (tuple, list)):
                embedding = embedding[0]
            
            # Convert to numpy and L2-normalize
            emb_np = embedding.cpu().numpy().flatten()
            
            # L2 normalize
            norm = np.linalg.norm(emb_np)
            if norm > 0:
                emb_np = emb_np / norm
            
            return emb_np.astype(np.float32)
            
        except Exception as e:
            print(f"      ⚠️ ViT embedding failed: {e}")
            return None

# Initialize ViT recognizer
vit_recognizer = ViTFaceRecognizer(config)

# ─── Monkey-patch get_embedding_direct (IDEMPOTENT — safe to re-run) ───
# Save the REAL original only once. If already patched, don't re-wrap.
if not hasattr(detector, '_original_unpatched_get_embedding_direct'):
    detector._original_unpatched_get_embedding_direct = detector.get_embedding_direct

def _vit_enhanced_get_embedding_direct(aligned_face: np.ndarray, keypoints: np.ndarray = None) -> Optional[np.ndarray]:
    """
    Enhanced embedding extraction: ViT first, ArcFace fallback.
    """
    # Try ViT first (global attention, occlusion-robust)
    if vit_recognizer.available:
        vit_emb = vit_recognizer.get_embedding(aligned_face, keypoints)
        if vit_emb is not None:
            return vit_emb
    
    # Fallback to ArcFace ResNet-50 (ALWAYS call the REAL original)
    return detector._original_unpatched_get_embedding_direct(aligned_face)

detector.get_embedding_direct = _vit_enhanced_get_embedding_direct
print("✅ ViT-enhanced recognition ready!" if vit_recognizer.available 
      else "✅ Using ArcFace ResNet-50 recognition (ViT unavailable)")

In [ ]:
# %% Cell 4: Face Alignment & Preprocessing
# ============================================================================
# CELL 4: FACE ALIGNMENT & PREPROCESSING
# ============================================================================
# Critical step: 5-point landmark affine transformation to 112×112.
# Without proper alignment, recognition accuracy drops 10-30%.
# ============================================================================

class FaceAligner:
    """
    Aligns detected faces to a canonical 112×112 format using 5-point landmarks.
    
    This is NOT just cropping the bounding box. It performs:
    1. Similarity transformation (rotation + scale) to align eyes horizontally
    2. Centers the nose
    3. Produces a standardized input for the recognition model
    
    Additionally applies CLAHE on LAB L-channel for illumination normalization.
    """
    
    # Standard alignment coordinates for 112×112 (ArcFace standard)
    REFERENCE_LANDMARKS = np.array([
        [38.2946, 51.6963],   # Left eye
        [73.5318, 51.5014],   # Right eye
        [56.0252, 71.7366],   # Nose tip
        [41.5493, 92.3655],   # Left mouth corner
        [70.7299, 92.2041],   # Right mouth corner
    ], dtype=np.float32)
    
    def __init__(self, output_size: int = 112, apply_clahe: bool = True):
        self.output_size = output_size
        self.apply_clahe = apply_clahe
        if apply_clahe:
            self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        print(f"   ✅ Aligner ready (output: {output_size}×{output_size}, CLAHE: {apply_clahe})")
    
    def align(self, frame: np.ndarray, landmarks: np.ndarray) -> np.ndarray:
        """
        Align face using 5-point landmarks via similarity transform.
        
        Args:
            frame: Original full frame (BGR)
            landmarks: 5×2 array of facial landmarks
            
        Returns:
            Aligned face chip, 112×112×3, BGR, uint8
        """
        # Estimate similarity transform
        M = self._estimate_similarity_transform(landmarks, self.REFERENCE_LANDMARKS)
        aligned = cv2.warpAffine(frame, M, (self.output_size, self.output_size))
        
        # Apply CLAHE on L-channel for illumination normalization
        if self.apply_clahe:
            aligned = self._apply_clahe(aligned)
        
        return aligned
    
    def _estimate_similarity_transform(self, src: np.ndarray, dst: np.ndarray) -> np.ndarray:
        """
        Estimate 2D similarity transformation matrix.
        Uses least-squares to find rotation, scale, and translation.
        """
        num = src.shape[0]
        dim = src.shape[1]
        
        src_mean = src.mean(axis=0)
        dst_mean = dst.mean(axis=0)
        
        src_demean = src - src_mean
        dst_demean = dst - dst_mean
        
        A = np.zeros((num * dim, 4))
        b = np.zeros(num * dim)
        
        for i in range(num):
            A[2*i]     = [src_demean[i, 0], -src_demean[i, 1], 1, 0]
            A[2*i + 1] = [src_demean[i, 1],  src_demean[i, 0], 0, 1]
            b[2*i]     = dst_demean[i, 0]
            b[2*i + 1] = dst_demean[i, 1]
        
        result, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        
        M = np.array([
            [result[0], -result[1], result[2] + dst_mean[0] - result[0]*src_mean[0] + result[1]*src_mean[1]],
            [result[1],  result[0], result[3] + dst_mean[1] - result[1]*src_mean[0] - result[0]*src_mean[1]]
        ])
        
        return M
    
    def _apply_clahe(self, image: np.ndarray) -> np.ndarray:
        """
        Apply CLAHE on LAB L-channel for illumination normalization.
        Does NOT use histogram equalization on RGB (which distorts colors).
        """
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l_channel, a_channel, b_channel = cv2.split(lab)
        l_channel = self.clahe.apply(l_channel)
        lab = cv2.merge([l_channel, a_channel, b_channel])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

aligner = FaceAligner(output_size=112, apply_clahe=True)
print("✅ Face aligner initialized!")

In [ ]:
# %% Cell 5: Face Recognition Module
# ============================================================================
# CELL 5: FACE RECOGNITION
# ============================================================================
# Using InsightFace's bundled recognition model (ResNet-100 ArcFace).
# This serves as our baseline. If LVFace ViT weights become available,
# they can be swapped in as a drop-in replacement (same 512-d output).
# ============================================================================

class FaceRecognizer:
    """
    Face recognition via embedding extraction.
    
    The InsightFace `buffalo_l` pack already computes embeddings during
    detection (.embedding attribute). This class provides:
    
    1. Standalone embedding extraction (for enrollment photos)
    2. L2 normalization (critical for cosine similarity = dot product)
    3. Quality-aware matching threshold
    
    Architecture notes:
    - Current: ResNet-100 ArcFace (Glint360k) → 512-d embeddings
    - Future upgrade: LVFace ViT → 512-d embeddings (drop-in replacement)
    - The ViT's self-attention handles occlusion better by dynamically
      weighting reliable face regions over corrupted ones.
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.embedding_dim = config.EMBEDDING_DIM
        # The recognition model is already loaded inside the FaceAnalysis app
        # We'll use the detector's .get() which returns faces with .embedding
        print(f"   ✅ Recognizer ready (dim: {self.embedding_dim})")
    
    @staticmethod
    def normalize(embedding: np.ndarray) -> np.ndarray:
        """
        L2-normalize embedding so that dot product = cosine similarity.
        
        For normalized vectors: A·B = ||A||||B||cos(θ) = 1·1·cos(θ) = cos(θ)
        """
        norm = np.linalg.norm(embedding)
        if norm == 0:
            return embedding
        return embedding / norm
    
    @staticmethod
    def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        """Compute cosine similarity between two embeddings."""
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))
    
    def compute_quality_score(self, face) -> float:
        """
        Estimate face quality from detection metadata.
        
        Uses face size and detection confidence as quality proxy.
        Higher quality → lower threshold needed for matching.
        Lower quality → demand higher similarity to confirm.
        """
        bbox = face.bbox
        face_width = bbox[2] - bbox[0]
        face_height = bbox[3] - bbox[1]
        face_area = face_width * face_height
        
        # Normalize: assume 50×50 = min usable, 200×200 = ideal
        size_score = np.clip(face_area / (200 * 200), 0, 1)
        
        # Detection confidence
        conf_score = float(face.det_score)
        
        # Combined quality
        quality = 0.5 * size_score + 0.5 * conf_score
        return float(np.clip(quality, 0, 1))
    
    def get_dynamic_threshold(self, quality_score: float) -> float:
        """
        Quality-aware dynamic threshold.
        
        Formula: T = T_base + λ * (1 - Q_score)
        
        - High quality face (Q=1.0): T = 0.4 (accept lower similarity)
        - Low quality face (Q=0.0): T = 0.55 (demand higher similarity)
        """
        return self.config.MATCH_THRESHOLD_BASE + \
               self.config.MATCH_THRESHOLD_LAMBDA * (1.0 - quality_score)

recognizer = FaceRecognizer(config)
print("✅ Face recognizer initialized!")

In [ ]:
# %% Cell 6: Generative Gallery Augmentation (GGA)
# ============================================================================
# CELL 6: GENERATIVE GALLERY AUGMENTATION
# ============================================================================
# The "killer feature": Generate synthetic appearance variations from a
# single enrollment photo to handle biological changes (beard, glasses, etc.)
#
# Primary: InstantID (identity-preserving diffusion)
# Fallback: Traditional augmentation (rotation, brightness, blur)
# ============================================================================

class TraditionalAugmentor:
    """
    Fallback augmentation when InstantID is unavailable or fails.
    
    Generates geometric and photometric variations of the enrollment photo.
    Not as powerful as generative augmentation, but still significantly
    improves matching by creating a multi-vector gallery.
    """
    
    def __init__(self):
        print("   📸 Traditional augmentor initialized")
    
    def generate_variations(self, face_chip: np.ndarray) -> List[Tuple[str, np.ndarray]]:
        """
        Generate augmented versions of a 112×112 face chip.
        
        Returns:
            List of (variant_name, augmented_image) tuples
        """
        variations = []
        h, w = face_chip.shape[:2]
        
        # 1. Slight rotation left (simulates head tilt)
        M = cv2.getRotationMatrix2D((w//2, h//2), 10, 1.0)
        rotated_left = cv2.warpAffine(face_chip, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        variations.append(("rotated_left", rotated_left))
        
        # 2. Slight rotation right
        M = cv2.getRotationMatrix2D((w//2, h//2), -10, 1.0)
        rotated_right = cv2.warpAffine(face_chip, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        variations.append(("rotated_right", rotated_right))
        
        # 3. Darkened (simulates shadow/low light)
        darkened = cv2.convertScaleAbs(face_chip, alpha=0.6, beta=-20)
        variations.append(("darkened", darkened))
        
        # 4. Brightened (simulates overexposure)
        brightened = cv2.convertScaleAbs(face_chip, alpha=1.3, beta=30)
        variations.append(("brightened", brightened))
        
        # 5. Slight blur (simulates low-res / motion)
        blurred = cv2.GaussianBlur(face_chip, (5, 5), 1.5)
        variations.append(("blurred", blurred))
        
        # 6. Horizontal flip (mirror)
        flipped = cv2.flip(face_chip, 1)
        variations.append(("flipped", flipped))
        
        # 7. Contrast enhancement
        lab = cv2.cvtColor(face_chip, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(4, 4))
        l = clahe.apply(l)
        enhanced = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
        variations.append(("contrast_enhanced", enhanced))
        
        # 8. Partial occlusion simulation (lower face masked — simulates mask/hand)
        occluded = face_chip.copy()
        occluded[h//2:, :] = cv2.GaussianBlur(occluded[h//2:, :], (21, 21), 10)
        variations.append(("lower_occluded", occluded))
        
        return variations


class GenerativeAugmentor:
    """
    InstantID-based generative gallery augmentation.
    
    Uses SDXL + ControlNet + IP-Adapter to generate photorealistic
    variations of a face while preserving identity. This enables the
    system to mathematically anticipate appearance changes (beard, glasses)
    BEFORE they happen in the real world.
    
    If InstantID fails to load (VRAM/dependency issues), falls back
    to TraditionalAugmentor automatically.
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.pipeline = None
        self.available = False
        self.fallback = TraditionalAugmentor()
        
        if config.ENABLE_GGA:
            self._try_load_instantid()
    
    def _try_load_instantid(self):
        """Attempt to load InstantID pipeline. Fails gracefully."""
        try:
            import torch
            from huggingface_hub import hf_hub_download
            from diffusers import ControlNetModel, DDIMScheduler
            import diffusers
            
            print("🎨 Attempting to load InstantID pipeline...")
            print(f"   📦 diffusers version: {diffusers.__version__}")
            print("   ⏳ This may take a few minutes on first run...")
            
            # Check VRAM
            if torch.cuda.is_available():
                vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
                print(f"   GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB)")
                
                if vram_gb < 14.0:  # Relaxed slightly for Kaggle T4 which sometimes shows 14.6
                    print("   ⚠️  Less than 14.0GB VRAM — InstantID may OOM.")
                    print("   📸 Using traditional augmentation as primary method.")
                    return
            
            # Download ControlNet and IP-Adapter weights first
            print("   📥 Downloading InstantID weights...")
            controlnet = ControlNetModel.from_pretrained(
                "InstantX/InstantID",
                subfolder="ControlNetModel",
                torch_dtype=torch.float16,
            )
            
            face_adapter_path = hf_hub_download(
                "InstantX/InstantID",
                filename="ip-adapter.bin"
            )
            print(f"      ✅ IP-Adapter downloaded: {face_adapter_path}")
            
            # Download or load the community InstantID pipeline
            try:
                import sys
                import os
                import urllib.request
                
                script_name = "pipeline_stable_diffusion_xl_instantid.py"
                if not os.path.exists(script_name):
                    print("   📥 Downloading community pipeline script...")
                    url = "https://raw.githubusercontent.com/huggingface/diffusers/main/examples/community/pipeline_stable_diffusion_xl_instantid.py"
                    urllib.request.urlretrieve(url, script_name)
                    print("      ✅ Script downloaded.")
                
                # --- AUTO-PATCH THE DOWNLOADED SCRIPT FOR TENSOR DEVICE MISMATCH ---
                # image_proj_model is not managed by diffusers offload, causing it to crash 
                # against CUDA inputs. We dynamically rewrite the script on disk before importing.
                print("   🔧 Auto-patching community script for CPU offload bugs...")
                with open(script_name, "r", encoding="utf-8") as f:
                    script_content = f.read()
                
                bugged_line = "prompt_image_emb = self.image_proj_model(prompt_image_emb)"
                patched_line = "self.image_proj_model.to(device=device, dtype=dtype)\n        prompt_image_emb = self.image_proj_model(prompt_image_emb)"
                
                if bugged_line in script_content and patched_line not in script_content:
                    script_content = script_content.replace(bugged_line, patched_line)
                    with open(script_name, "w", encoding="utf-8") as f:
                        f.write(script_content)
                    print("      ✅ Patch applied successfully.")
                
                # Ensure current directory is in path to import the local pipeline script
                if os.getcwd() not in sys.path:
                    sys.path.insert(0, os.getcwd())
                    
                # Force reload if it was already imported previously in the kernel
                if 'pipeline_stable_diffusion_xl_instantid' in sys.modules:
                    del sys.modules['pipeline_stable_diffusion_xl_instantid']
                    
                from pipeline_stable_diffusion_xl_instantid import StableDiffusionXLInstantIDPipeline, draw_kps
                print("   🔄 Loading InstantID pipeline via community script...")
                
                # --- MONKEYPATCH DIFFUSERS BUG --- 
                # Diffusers 0.36.0 has a bug in its base StableDiffusionXLControlNetPipeline.check_inputs 
                # where it incorrectly identifies InstantID as a MultiControlNet and crashes on the scale type.
                # We bypass this logic by replacing the validation method.
                original_check = StableDiffusionXLInstantIDPipeline.check_inputs
                def patched_check_inputs(self, *args, **kwargs):
                    try:
                        # Attempt original validation
                        original_check(self, *args, **kwargs)
                    except Exception as e:
                        error_msg = str(e)
                        is_scale_error = "controlnet_conditioning_scale" in error_msg
                        is_type_error = "must be type" in error_msg or "subscriptable" in error_msg or "iterable" in error_msg
                        
                        if is_scale_error or (is_type_error and "float" in error_msg):
                            pass  # Ignore diffusers 0.36.0 ControlNet validation bugs
                        else:
                            raise e
                            
                StableDiffusionXLInstantIDPipeline.check_inputs = patched_check_inputs
                # ----------------------------------
                
                self.pipeline = StableDiffusionXLInstantIDPipeline.from_pretrained(
                    "stabilityai/stable-diffusion-xl-base-1.0",
                    controlnet=controlnet,
                    torch_dtype=torch.float16,
                    variant="fp16",
                )
                
                self.pipeline.load_ip_adapter_instantid(face_adapter_path)
                self.pipeline.set_ip_adapter_scale(0.8)
                
                self.pipeline.scheduler = DDIMScheduler.from_config(
                    self.pipeline.scheduler.config
                )
                
                self.pipeline.enable_model_cpu_offload()
                self.pipeline_type = "instantid"
                self.available = True
                print("   ✅ InstantID pipeline loaded via community script!")
                
                # Store draw_kps function for later use
                self.draw_kps = draw_kps
                
            except Exception as e:
                print(f"   ⚠️ Community InstantID pipeline failed: {e}")
                self.available = False
            
        except Exception as e:
            print(f"   ⚠️  InstantID failed to load: {e}")
            import traceback
            traceback.print_exc()
            print("   📸 Falling back to traditional augmentation.")
            self.available = False
    
    def generate_variations(
        self, 
        full_image: np.ndarray,
        aligned_chip: np.ndarray,
        person_name: str
    ) -> List[Tuple[str, np.ndarray]]:
        """
        Generate identity-preserving variations of an enrollment face.
        
        Tries InstantID first (needs full_image), falls back to traditional 
        augmentation (needs aligned_chip).
        
        Args:
            full_image: Original uncropped high-res image (BGR)
            aligned_chip: Aligned 112×112 face chip (BGR)
            person_name: For logging purposes
            
        Returns:
            List of (variant_name, face_chip) tuples
        """
        if not self.available:
            print(f"   ⚠️  [DEBUG] Skipping InstantID for {person_name}: self.available is False!")
        if self.pipeline is None:
            print(f"   ⚠️  [DEBUG] Skipping InstantID for {person_name}: self.pipeline is None! (Cleanup was likely called)")
            
        if self.available and self.pipeline is not None:
            try:
                return self._generate_instantid(full_image, person_name)
            except Exception as e:
                print(f"   ⚠️  InstantID generation failed for {person_name}: {e}")
                if self.config.GGA_FALLBACK:
                    print(f"   📸 Falling back to traditional augmentation for {person_name}")
        
        # Fallback to traditional augmentation
        return self.fallback.generate_variations(aligned_chip)
    
    def _generate_instantid(
        self, 
        full_image: np.ndarray,
        person_name: str
    ) -> List[Tuple[str, np.ndarray]]:
        """Generate variations using InstantID."""
        import torch
        from PIL import Image
        
        # Convert full image from BGR to RGB PIL Image
        # InstantID needs a decent resolution (e.g., 512-1024) to generate 
        # high-quality faces, rather than upscaling a tiny 112x112 chip.
        img_rgb = cv2.cvtColor(full_image, cv2.COLOR_BGR2RGB)
        
        # Smart resize: Keep aspect ratio but ensure min dimension is ~640
        h, w = img_rgb.shape[:2]
        scale = 640 / min(h, w)
        if scale < 1.0: # Only downscale to save memory
            new_h, new_w = int(h * scale), int(w * scale)
            img_rgb = cv2.resize(img_rgb, (new_w, new_h))
            
        face_pil = Image.fromarray(img_rgb)
        
        # We also need to extract embedding/kps using our detector on the resized image
        face_info = detector.detect(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
        if face_info and len(face_info) > 0:
            face_emb = face_info[0].embedding
            face_kps = face_info[0].kps
        else:
            print(f"   ⚠️  [DEBUG] detector.detect() found NO FACES in the resized image for {person_name}!")
            # If no face detected in chip, use a zero embedding as fallback
            face_emb = np.zeros(512, dtype=np.float32)
            face_kps = None
        
        variations = []
        
        if face_kps is not None:
            # InstantID requires face keypoints drawn as an image for ControlNet input
            # Create the keypoint image
            face_kps_img = self.draw_kps(face_pil, face_kps)
            
            for i, prompt in enumerate(self.config.GGA_PROMPTS):
                print(f"   🎨 Generating variation {i+1}/{len(self.config.GGA_PROMPTS)}: {prompt[:50]}...")
                
                try:
                    with torch.no_grad():
                        result = self.pipeline(
                            prompt=prompt,
                            negative_prompt="blurry, low quality, distorted, deformed",
                            image_embeds=face_emb,
                            image=face_kps_img,
                            controlnet_conditioning_scale=float(0.8),
                            num_inference_steps=25,
                            guidance_scale=5.0,
                        ).images[0]
                    
                    # Convert back to 112×112 BGR numpy (for Gallery storage)
                    result_np_112 = np.array(result.resize((112, 112)))
                    result_bgr_112 = cv2.cvtColor(result_np_112, cv2.COLOR_RGB2BGR)
                    
                    variant_name = f"gen_{i}"
                    variations.append((variant_name, result_bgr_112))
                    
                    # Cache the FULL-RES HIGH-QUALITY image to disk for visualization later
                    try:
                        from pathlib import Path
                        import os
                        vis_dir = Path(self.config.OUTPUT_DIR) / "gga_vis" / person_name
                        vis_dir.mkdir(parents=True, exist_ok=True)
                        
                        # Convert full-res PIL to BGR for cv2
                        result_np_full = np.array(result)
                        result_bgr_full = cv2.cvtColor(result_np_full, cv2.COLOR_RGB2BGR)
                        
                        save_path = str(vis_dir / f"{variant_name}.jpg")
                        cv2.imwrite(save_path, result_bgr_full)
                        print(f"      💾 Saved cache: {save_path}")
                    except Exception as save_err:
                        print(f"      ❌ Saving cache failed: {save_err}")
                        
                except Exception as e:
                    print(f"      ⚠️ Generation {i} failed: {e}")
        
        return variations
    
    def cleanup(self):
        """Free GPU memory after enrollment is complete."""
        if self.pipeline is not None:
            import torch
            import gc
            
            # Explicitly delete the heavy pipelines and components
            del self.pipeline
            self.pipeline = None
            
            if hasattr(self, '_ip_adapter_state'):
                del self._ip_adapter_state
            
            # Force garbage collection
            gc.collect()
            
            # Empty CUDA caches
            torch.cuda.empty_cache()
            if torch.cuda.is_available():
                torch.cuda.ipc_collect()
                
            print("   🗑️  InstantID pipeline forcefully purged from GPU memory.")

augmentor = GenerativeAugmentor(config)
print("✅ Augmentor initialized!")

In [ ]:
# %% Cell 7: FAISS Index (Exact Search)
# ============================================================================
# CELL 7: FAISS IndexFlatIP — EXACT VECTOR SEARCH
# ============================================================================
# At <5,000 vectors, brute-force dot product takes MICROSECONDS on CPU.
# No need for approximate search (HNSW/IVF) — exact search guarantees
# 100% recall with zero complexity overhead.
# ============================================================================

class GalleryIndex:
    """
    FAISS-based gallery index for face matching.
    
    Uses IndexFlatIP (Inner Product) with L2-normalized vectors.
    For normalized vectors: dot_product(A, B) = cosine_similarity(A, B)
    
    Why IndexFlatIP over HNSW/IVF:
    - At N<5,000 vectors, brute-force is actually FASTER (no graph traversal)
    - Guarantees 100% recall (no approximate search errors)
    - Zero tuning parameters (no nprobe, efSearch, etc.)
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.dim = config.EMBEDDING_DIM
        self.index = faiss.IndexFlatIP(self.dim)
        
        # Metadata: maps vector index → (person_name, variant_type)
        self.id_map: List[Tuple[str, str]] = []
        self.person_names: set = set()
        
        print(f"   ✅ FAISS IndexFlatIP created (dim={self.dim})")
    
    def add_embedding(self, embedding: np.ndarray, person_name: str, variant: str = "original"):
        """
        Add a single L2-normalized embedding to the index.
        
        Args:
            embedding: 512-d embedding vector
            person_name: Identity label
            variant: Type of image (original, gen_0, rotated_left, etc.)
        """
        # Ensure L2 normalization
        normalized = FaceRecognizer.normalize(embedding)
        
        # FAISS requires float32 and 2D array
        vec = normalized.astype(np.float32).reshape(1, -1)
        self.index.add(vec)
        self.id_map.append((person_name, variant))
        self.person_names.add(person_name)
    
    def search(self, query_embedding: np.ndarray, k: int = 1) -> List[Tuple[str, float, str]]:
        """
        Search for the nearest identity.
        
        Args:
            query_embedding: 512-d embedding (will be normalized)
            k: Number of nearest neighbors
            
        Returns:
            List of (person_name, similarity_score, variant_type)
        """
        # Normalize query
        normalized = FaceRecognizer.normalize(query_embedding)
        query = normalized.astype(np.float32).reshape(1, -1)
        
        # Search
        similarities, indices = self.index.search(query, k)
        
        results = []
        for sim, idx in zip(similarities[0], indices[0]):
            if idx < 0 or idx >= len(self.id_map):
                continue
            person_name, variant = self.id_map[idx]
            results.append((person_name, float(sim), variant))
        
        return results
    
    def match(self, query_embedding: np.ndarray, quality_score: float = 1.0) -> Tuple[str, float]:
        """
        Match a face embedding against the gallery with dynamic threshold.
        
        Args:
            query_embedding: 512-d embedding
            quality_score: Face quality score (0-1)
            
        Returns:
            (person_name, similarity) or (UNKNOWN_LABEL, similarity)
        """
        if self.index.ntotal == 0:
            return (self.config.UNKNOWN_LABEL, 0.0)
        
        results = self.search(query_embedding, k=3)
        
        if not results:
            return (self.config.UNKNOWN_LABEL, 0.0)
        
        best_name, best_sim, best_variant = results[0]
        
        # Dynamic threshold based on quality
        threshold = recognizer.get_dynamic_threshold(quality_score)
        
        if best_sim >= threshold:
            return (best_name, best_sim)
        else:
            return (self.config.UNKNOWN_LABEL, best_sim)
    
    def get_stats(self) -> dict:
        """Return index statistics."""
        person_counts = Counter(name for name, _ in self.id_map)
        return {
            "total_vectors": self.index.ntotal,
            "total_persons": len(self.person_names),
            "persons": dict(person_counts),
        }

gallery = GalleryIndex(config)
print("✅ FAISS gallery index initialized!")

In [ ]:
# %% Cell 8: Enrollment Pipeline
# ============================================================================
# CELL 8: ENROLLMENT PIPELINE
# ============================================================================
# Processes all enrollment photos:
# 1. Detect face → 2. Align → 3. Extract embedding → 4. Generate variations
# → 5. Extract variation embeddings → 6. Add all to FAISS index
# ============================================================================

class EnrollmentPipeline:
    """
    End-to-end enrollment: Single photo → Multi-vector gallery entry.
    
    For each enrollment photo:
    - Detects and aligns the face
    - Extracts the original embedding
    - Generates synthetic variations (GGA)
    - Extracts embeddings from all variations
    - Adds everything to the FAISS index
    
    The synthetic images are TRANSIENT — only embeddings are stored.
    """
    
    def __init__(
        self,
        detector: FaceDetector,
        aligner: FaceAligner,
        recognizer: FaceRecognizer,
        augmentor: GenerativeAugmentor,
        gallery: GalleryIndex,
        config: Config,
    ):
        self.detector = detector
        self.aligner = aligner
        self.recognizer = recognizer
        self.augmentor = augmentor
        self.gallery = gallery
        self.config = config
    
    def enroll_person(self, image_path: str, person_name: str) -> bool:
        """
        Enroll a single person from their photo.
        
        Args:
            image_path: Path to the enrollment photo
            person_name: Identity label
            
        Returns:
            True if enrollment succeeded
        """
        print(f"\n👤 Enrolling: {person_name}")
        print(f"   📷 Image: {image_path}")
        
        # Load image
        image = cv2.imread(image_path)
        if image is None:
            print(f"   ❌ Failed to load image: {image_path}")
            return False
        
        # Detect face
        face = self.detector.detect_for_enrollment(image)
        if face is None:
            print(f"   ❌ No face detected in {image_path}")
            return False
        
        print(f"   ✅ Face detected (confidence: {face.det_score:.3f})")
        
        # Get aligned face chip
        if face.kps is not None:
            aligned = self.aligner.align(image, face.kps)
        else:
            # Fallback: crop bounding box and resize
            bbox = face.bbox.astype(int)
            crop = image[bbox[1]:bbox[3], bbox[0]:bbox[2]]
            aligned = cv2.resize(crop, (112, 112))
            print("   ⚠️  No landmarks — using crop fallback (accuracy reduced)")
        
        # Extract original embedding — prefer ViT (global attention, occlusion-robust)
        if vit_recognizer.available:
            kps = face.kps if face.kps is not None else None
            original_embedding = vit_recognizer.get_embedding(aligned, kps)
            if original_embedding is not None:
                print(f"   🧠 Using ViT embedding for {person_name}")
            else:
                original_embedding = face.embedding
                print(f"   📸 ViT failed, using ArcFace for {person_name}")
        else:
            original_embedding = face.embedding
        
        if original_embedding is None:
            print(f"   ❌ No embedding extracted for {person_name}")
            return False
        
        # Add original to gallery
        self.gallery.add_embedding(original_embedding, person_name, "original")
        print(f"   ✅ Original embedding added (dim={len(original_embedding)})")
        print(f"      📊 Embedding stats: mean={original_embedding.mean():.4f}, std={original_embedding.std():.4f}, norm={np.linalg.norm(original_embedding):.4f}")
        
        # Generate and enroll variations
        # CRITICAL: Pass BOTH the full original image (for InstantID) 
        # AND the aligned chip (for traditional fallback)
        variations = self.augmentor.generate_variations(image, aligned, person_name)
        print(f"   🎨 Generated {len(variations)} variations")
        
        enrolled_variants = 0
        for variant_name, variant_chip in variations:
            # USE DIRECT EMBEDDING EXTRACTION (bypass detection)
            # This is critical: SCRFD can't reliably detect faces in 112×112 chips
            # Most variants were being SILENTLY DROPPED before this fix.
            var_embedding = self.detector.get_embedding_direct(variant_chip)
            if var_embedding is not None:
                self.gallery.add_embedding(var_embedding, person_name, variant_name)
                enrolled_variants += 1
            else:
                # Fallback: try detection on padded version, but STILL use ViT
                padded = cv2.copyMakeBorder(variant_chip, 40, 40, 40, 40, 
                                           cv2.BORDER_REPLICATE)
                var_faces = self.detector.detect(padded)
                if var_faces:
                    best_var = max(var_faces, key=lambda f: f.det_score)
                    if best_var.kps is not None:
                        var_aligned = self.aligner.align(padded, best_var.kps)
                        var_emb = vit_recognizer.get_embedding(var_aligned) if vit_recognizer.available else best_var.embedding
                    else:
                        var_emb = best_var.embedding
                    if var_emb is not None:
                        self.gallery.add_embedding(var_emb, person_name, variant_name)
                        enrolled_variants += 1
        
        print(f"   ✅ Enrolled {enrolled_variants + 1} vectors total for {person_name}")
        
        # Also enroll at different scales of the ORIGINAL image for robustness
        # CRITICAL: Must use ViT embeddings here too (same space as gallery)
        for scale_name, scale in [("zoomed_in", 1.2), ("zoomed_out", 0.8)]:
            h, w = image.shape[:2]
            scaled = cv2.resize(image, (int(w*scale), int(h*scale)))
            scaled_faces = self.detector.detect(scaled)
            if scaled_faces:
                best_face = max(scaled_faces, key=lambda f: f.det_score)
                if best_face.kps is not None:
                    scaled_aligned = self.aligner.align(scaled, best_face.kps)
                    if vit_recognizer.available:
                        scaled_emb = vit_recognizer.get_embedding(scaled_aligned)
                    else:
                        scaled_emb = best_face.embedding
                    if scaled_emb is not None:
                        self.gallery.add_embedding(scaled_emb, person_name, scale_name)
                        enrolled_variants += 1
        
        print(f"   ✅ Final total: {enrolled_variants + 1} vectors for {person_name}")
        return True
    
    def enroll_directory(self, directory: str) -> dict:
        """
        Enroll all persons from a directory of photos.
        
        Expected structure:
            directory/
                person1.jpg
                person2.jpg
                ...
        
        Person name is derived from the filename (without extension).
        """
        print("\n" + "="*60)
        print("📋 ENROLLMENT PIPELINE")
        print("="*60)
        
        directory = Path(directory)
        if not directory.exists():
            print(f"❌ Directory not found: {directory}")
            return {"success": 0, "failed": 0, "total": 0}
        
        # Find all image files
        image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        image_files = [
            f for f in sorted(directory.iterdir()) 
            if f.suffix.lower() in image_extensions
        ]
        
        print(f"📁 Found {len(image_files)} images in {directory}")
        
        results = {"success": 0, "failed": 0, "total": len(image_files), "persons": []}
        
        for img_file in image_files:
            person_name = img_file.stem  # filename without extension
            success = self.enroll_person(str(img_file), person_name)
            
            if success:
                results["success"] += 1
                results["persons"].append(person_name)
            else:
                results["failed"] += 1
        
        # Free InstantID from GPU memory (no longer needed)
        self.augmentor.cleanup()
        
        print("\n" + "="*60)
        print(f"📊 ENROLLMENT COMPLETE")
        print(f"   ✅ Success: {results['success']}/{results['total']}")
        print(f"   ❌ Failed:  {results['failed']}/{results['total']}")
        print(f"   📊 Gallery: {self.gallery.get_stats()}")
        print("="*60)
        
        return results

enrollment = EnrollmentPipeline(detector, aligner, recognizer, augmentor, gallery, config)
print("✅ Enrollment pipeline ready!")

# Run the actual enrollment process!
enrollment_results = enrollment.enroll_directory(config.ENROLLMENT_DIR)


In [ ]:
# %% Cell 8B: Visualize GGA Results (Optional)
# ============================================================================
# CELL 8B: VISUALIZE GENERATIVE GALLERY AUGMENTATION
# ============================================================================
# Run this cell after enrollment to visually inspect the 
# generated variations for a specific person.
# ============================================================================
import matplotlib.pyplot as plt

def visualize_gga_results(person_name: str, config: Config):
    """Visualizes the generated variants stored on disk."""
    if not config.ENABLE_GGA:
        print("⚠️ GGA is not enabled. Only traditional variants exist.")
        return
        
    print(f"\n🔍 Visualizing variations for: {person_name}")
    
    from pathlib import Path
    import matplotlib.pyplot as plt
    
    vis_dir = Path(config.OUTPUT_DIR) / "gga_vis" / person_name
    if not vis_dir.exists():
        print(f"❌ No cached generative images found for {person_name} in {vis_dir}")
        print("   Make sure you run the Enrollment Cell 8 first so they are saved!")
        return
        
    # Load original image
    enrollment_dir = Path(config.ENROLLMENT_DIR)
    orig_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        candidate = enrollment_dir / f"{person_name}{ext}"
        if candidate.exists():
            orig_path = candidate
            break
            
    # Load variant images
    variant_files = list(vis_dir.glob("gen_*.jpg"))
    if not variant_files:
        print(f"❌ No variant images found in {vis_dir}")
        return
        
    num_images = len(variant_files) + (1 if orig_path else 0)
    fig, axes = plt.subplots(1, num_images, figsize=(4 * num_images, 4))
    
    if num_images == 1:
        axes = [axes]
        
    idx = 0
    if orig_path:
        # Original
        orig_img = plt.imread(str(orig_path))
        axes[idx].imshow(orig_img)
        axes[idx].set_title("Original")
        axes[idx].axis('off')
        idx += 1
        
    for var_file in sorted(variant_files):
        var_img = plt.imread(str(var_file))
        axes[idx].imshow(var_img)
        axes[idx].set_title(var_file.stem)
        axes[idx].axis('off')
        idx += 1
        
    plt.tight_layout()
    plt.show()

# Example usage (Uncomment to test after enrollment):
visualize_gga_results("aaditya_singhal", config)

In [ ]:
# %% Cell 9: ByteTrack Multi-Object Tracker
# ============================================================================
# CELL 9: BYTETRACK MULTI-OBJECT TRACKER
# ============================================================================
# Tracks face detections across frames to maintain consistent identity.
# Key advantage over DeepSORT: ByteTrack uses LOW-confidence detections
# too, keeping track IDs alive during partial occlusion.
# ============================================================================

class IdentityTracker:
    """
    Multi-object face tracker with identity voting.
    
    Why ByteTrack over DeepSORT:
    - DeepSORT DISCARDS low-confidence detections (e.g., hand over face)
    - ByteTrack uses a two-stage association:
        1. Match high-confidence detections to tracks
        2. Match remaining LOW-confidence detections to unmatched tracks
    - This keeps the Track_ID alive during occlusion
    
    Identity voting:
    - A track is confirmed as "Person X" only if 3 out of 5 recent 
      recognition results agree. This prevents flickering labels.
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.tracker = sv.ByteTrack(
            track_activation_threshold=config.TRACK_THRESH,
            lost_track_buffer=config.TRACK_BUFFER,
            minimum_matching_threshold=config.MATCH_THRESH,
            frame_rate=30,
        )
        
        # Track_ID → list of recent identity votes
        self.identity_votes: Dict[int, List[str]] = defaultdict(list)
        # Track_ID → confirmed identity
        self.confirmed_identities: Dict[int, str] = {}
        
        print("   ✅ ByteTrack tracker initialized")
    
    def update(self, detections: sv.Detections) -> sv.Detections:
        """
        Update tracker with new detections.
        
        Args:
            detections: supervision.Detections object
            
        Returns:
            Tracked detections with tracker_id assigned
        """
        return self.tracker.update_with_detections(detections)
    
    def vote_identity(self, track_id: int, identity: str) -> str:
        """
        Submit an identity vote for a track and return the consensus.
        
        Implements temporal smoothing: The identity is only confirmed
        when VOTE_MIN out of VOTE_WINDOW recent votes agree.
        
        Args:
            track_id: The tracker-assigned ID
            identity: The recognition result for this frame
            
        Returns:
            The consensus identity (may differ from input if overridden by history)
        """
        votes = self.identity_votes[track_id]
        votes.append(identity)
        
        # Keep only last VOTE_WINDOW votes
        if len(votes) > self.config.VOTE_WINDOW:
            votes = votes[-self.config.VOTE_WINDOW:]
            self.identity_votes[track_id] = votes
        
        # Count votes (excluding "Unknown")
        known_votes = [v for v in votes if v != self.config.UNKNOWN_LABEL]
        
        if known_votes:
            counter = Counter(known_votes)
            best_name, best_count = counter.most_common(1)[0]
            
            if best_count >= self.config.VOTE_MIN:
                self.confirmed_identities[track_id] = best_name
                return best_name
        
        # Return previously confirmed identity if exists (temporal smoothing)
        if track_id in self.confirmed_identities:
            return self.confirmed_identities[track_id]
        
        return self.config.UNKNOWN_LABEL
    
    def get_identity(self, track_id: int) -> str:
        """Get the current confirmed identity for a track."""
        return self.confirmed_identities.get(track_id, self.config.UNKNOWN_LABEL)
    
    def get_all_confirmed(self) -> Dict[int, str]:
        """Get all confirmed identities."""
        return dict(self.confirmed_identities)

tracker = IdentityTracker(config)
print("✅ Identity tracker initialized!")

In [ ]:
# %% Cell 10: Video Inference Pipeline
# ============================================================================
# CELL 10: VIDEO INFERENCE PIPELINE
# ============================================================================
# The main loop: Reads video → Detects → Tracks → Recognizes → Annotates
#
# Optimization: Recognition runs every N frames (expensive).
# Tracking fills the gaps between recognition frames.
# ============================================================================

class AttendanceInference:
    """
    End-to-end video inference pipeline.
    
    Architecture:
        Frame → SCRFD (detect) → ByteTrack (track) → LVFace (recognize)
              → FAISS (match) → Vote (confirm) → Annotate → Output
    
    Key optimization: Recognition is expensive (~30ms per face).
    We run it every RECOGNIZE_EVERY_N_FRAMES frames. ByteTrack's
    tracking fills the gaps so we still see smooth bounding boxes.
    """
    
    def __init__(
        self,
        detector: FaceDetector,
        aligner: FaceAligner,
        recognizer: FaceRecognizer,
        gallery: GalleryIndex,
        tracker: IdentityTracker,
        config: Config,
    ):
        self.detector = detector
        self.aligner = aligner
        self.recognizer = recognizer
        self.gallery = gallery
        self.tracker = tracker
        self.config = config
        
        # Attendance log
        self.attendance: Dict[str, dict] = {}  # person_name → {first_seen, last_seen, count}
        self.frame_count = 0
    
    def _faces_to_detections(self, faces: list) -> sv.Detections:
        """Convert InsightFace face objects to supervision Detections."""
        if not faces:
            return sv.Detections.empty()
        
        bboxes = np.array([f.bbox for f in faces])
        confidences = np.array([f.det_score for f in faces])
        
        return sv.Detections(
            xyxy=bboxes,
            confidence=confidences,
        )
    
    def process_frame(self, frame: np.ndarray, do_recognize: bool = True) -> Tuple[np.ndarray, list]:
        """
        Process a single video frame.
        
        Args:
            frame: BGR video frame
            do_recognize: Whether to run recognition this frame
            
        Returns:
            (annotated_frame, list_of_detections_info)
        """
        self.frame_count += 1
        
        # 1. DETECT
        faces = self.detector.detect(frame)
        
        # 2. TRACK
        detections = self._faces_to_detections(faces)
        tracked = self.tracker.update(detections)
        
        # 3. RECOGNIZE (only every N frames for efficiency)
        frame_results = []
        
        if do_recognize and tracked.tracker_id is not None:
            for i, (bbox, track_id) in enumerate(zip(tracked.xyxy, tracked.tracker_id)):
                # Find the matching face object for this detection
                face = self._find_matching_face(faces, bbox)
                
                if face is not None:
                    # Get embedding — prefer ViT, fall back to ArcFace
                    embedding = None
                    if vit_recognizer.available and face.kps is not None:
                        # Align face for ViT input
                        aligned_chip = self.aligner.align(frame, face.kps)
                        embedding = vit_recognizer.get_embedding(aligned_chip, face.kps)
                    
                    # Fallback to ArcFace
                    if embedding is None:
                        embedding = face.embedding
                    
                    if embedding is not None:
                        # Compute quality
                        quality = self.recognizer.compute_quality_score(face)
                        
                        # Match against gallery
                        identity, similarity = self.gallery.match(embedding, quality)
                        
                        # Vote
                        confirmed = self.tracker.vote_identity(track_id, identity)
                        
                        frame_results.append({
                            "track_id": int(track_id),
                            "identity": confirmed,
                            "similarity": similarity,
                            "quality": quality,
                            "bbox": bbox.tolist(),
                        })
                        
                        # Update attendance
                        if confirmed != self.config.UNKNOWN_LABEL:
                            self._update_attendance(confirmed)
        
        # 4. ANNOTATE
        annotated = self._annotate_frame(frame, tracked)
        
        return annotated, frame_results
    
    def _find_matching_face(self, faces: list, bbox: np.ndarray, iou_thresh: float = 0.5):
        """Find the face object that best matches a tracked bounding box."""
        best_face = None
        best_iou = 0
        
        for face in faces:
            iou = self._compute_iou(face.bbox, bbox)
            if iou > best_iou:
                best_iou = iou
                best_face = face
        
        return best_face if best_iou >= iou_thresh else None
    
    @staticmethod
    def _compute_iou(box1: np.ndarray, box2: np.ndarray) -> float:
        """Compute Intersection over Union between two bounding boxes."""
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        intersection = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - intersection
        
        return intersection / union if union > 0 else 0
    
    def _update_attendance(self, person_name: str):
        """Update the attendance record for a person."""
        now = self.frame_count
        if person_name not in self.attendance:
            self.attendance[person_name] = {
                "first_seen_frame": now,
                "last_seen_frame": now,
                "detection_count": 1,
                "status": "PRESENT",
            }
        else:
            self.attendance[person_name]["last_seen_frame"] = now
            self.attendance[person_name]["detection_count"] += 1
    
    def _annotate_frame(self, frame: np.ndarray, tracked: sv.Detections) -> np.ndarray:
        """Draw bounding boxes and identity labels on the frame."""
        annotated = frame.copy()
        
        if tracked.tracker_id is None:
            return annotated
        
        for bbox, track_id in zip(tracked.xyxy, tracked.tracker_id):
            identity = self.tracker.get_identity(track_id)
            bbox_int = bbox.astype(int)
            
            # Color: green for known, red for unknown
            if identity == self.config.UNKNOWN_LABEL:
                color = self.config.UNKNOWN_COLOR
                label = f"Unknown (#{track_id})"
            else:
                color = self.config.BBOX_COLOR
                label = f"{identity} (#{track_id})"
            
            # Draw bbox
            cv2.rectangle(annotated, 
                         (bbox_int[0], bbox_int[1]), 
                         (bbox_int[2], bbox_int[3]),
                         color, 2)
            
            # Draw label background
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 
                                           self.config.FONT_SCALE, self.config.FONT_THICKNESS)
            cv2.rectangle(annotated,
                         (bbox_int[0], bbox_int[1] - th - 10),
                         (bbox_int[0] + tw, bbox_int[1]),
                         color, -1)
            
            # Draw label text
            cv2.putText(annotated, label,
                       (bbox_int[0], bbox_int[1] - 5),
                       cv2.FONT_HERSHEY_SIMPLEX,
                       self.config.FONT_SCALE,
                       (255, 255, 255),
                       self.config.FONT_THICKNESS)
        
        return annotated
    
    def run_on_video(self, video_path: str) -> dict:
        """
        Run the full attendance pipeline on a video file.
        
        Args:
            video_path: Path to the input video
            
        Returns:
            Results dictionary with attendance log, stats, and output paths
        """
        print("\n" + "="*60)
        print("🎥 VIDEO INFERENCE PIPELINE")
        print("="*60)
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"❌ Failed to open video: {video_path}")
            return {}
        
        # Video properties
        fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        print(f"   📐 Resolution: {width}×{height}")
        print(f"   🎬 FPS: {fps}, Total frames: {total_frames}")
        print(f"   ⏩ Processing every {self.config.PROCESS_EVERY_N_FRAMES}th frame")
        print(f"   🧠 Recognition every {self.config.RECOGNIZE_EVERY_N_FRAMES}th frame")
        
        # Output video writer
        out_path = os.path.join(self.config.OUTPUT_DIR, "annotated_output.mp4")
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out_fps = fps // self.config.PROCESS_EVERY_N_FRAMES
        writer = cv2.VideoWriter(out_path, fourcc, out_fps, (width, height))
        
        # Process frames
        frame_idx = 0
        processed = 0
        all_results = []
        start_time = time.time()
        
        frames_to_process = min(total_frames, self.config.MAX_FRAMES)
        
        while True:
            ret, frame = cap.read()
            if not ret or frame_idx >= frames_to_process:
                break
            
            frame_idx += 1
            
            # Skip frames for speed
            if frame_idx % self.config.PROCESS_EVERY_N_FRAMES != 0:
                continue
            
            # Determine if we should run recognition this frame
            do_recognize = (processed % (self.config.RECOGNIZE_EVERY_N_FRAMES // 
                                         self.config.PROCESS_EVERY_N_FRAMES) == 0)
            
            # Process
            annotated, results = self.process_frame(frame, do_recognize=do_recognize)
            writer.write(annotated)
            all_results.extend(results)
            processed += 1
            
            # Progress bar
            if processed % 10 == 0:
                elapsed = time.time() - start_time
                fps_actual = processed / elapsed if elapsed > 0 else 0
                progress = frame_idx / frames_to_process * 100
                print(f"   ⏳ Progress: {progress:.0f}% | Frame {frame_idx}/{frames_to_process} | "
                      f"{fps_actual:.1f} FPS | Faces tracked: {len(self.tracker.get_all_confirmed())}")
        
        cap.release()
        writer.release()
        
        elapsed = time.time() - start_time
        
        # Save sample annotated frames
        self._save_sample_frames(video_path, frames_to_process)
        
        print(f"\n   ✅ Processing complete!")
        print(f"   ⏱️  Time: {elapsed:.1f}s ({processed/elapsed:.1f} processed FPS)")
        print(f"   📹 Output video: {out_path}")
        
        return {
            "output_video": out_path,
            "total_frames": frame_idx,
            "processed_frames": processed,
            "elapsed_seconds": elapsed,
            "fps": processed / elapsed if elapsed > 0 else 0,
            "attendance": self.attendance,
            "all_detections": len(all_results),
        }
    
    def _save_sample_frames(self, video_path: str, total_frames: int, num_samples: int = 5):
        """Save a few annotated frames as images for quick visual inspection."""
        cap = cv2.VideoCapture(video_path)
        sample_indices = np.linspace(0, total_frames - 1, num_samples, dtype=int)
        
        for i, target_frame in enumerate(sample_indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
            ret, frame = cap.read()
            if ret:
                faces = self.detector.detect(frame)
                annotated = self.detector.visualize(frame, faces)
                
                # Add frame number
                cv2.putText(annotated, f"Frame: {target_frame}", (10, 30),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
                
                out_path = os.path.join(self.config.OUTPUT_DIR, f"sample_frame_{i}.jpg")
                cv2.imwrite(out_path, annotated)
        
        cap.release()
        print(f"   📸 Saved {num_samples} sample frames to {self.config.OUTPUT_DIR}")
    
    def get_attendance_report(self) -> str:
        """Generate a formatted attendance report."""
        report = []
        report.append("\n" + "="*60)
        report.append("📋 ATTENDANCE REPORT")
        report.append("="*60)
        
        if not self.attendance:
            report.append("   No students detected.")
            return "\n".join(report)
        
        report.append(f"\n{'Name':<20} {'Status':<10} {'Detections':<12} {'First Frame':<12} {'Last Frame':<12}")
        report.append("-" * 66)
        
        for name, data in sorted(self.attendance.items()):
            report.append(
                f"{name:<20} {data['status']:<10} {data['detection_count']:<12} "
                f"{data['first_seen_frame']:<12} {data['last_seen_frame']:<12}"
            )
        
        report.append("-" * 66)
        report.append(f"Total present: {len(self.attendance)}")
        
        # Check for enrolled but absent students
        enrolled = gallery.person_names
        present = set(self.attendance.keys())
        absent = enrolled - present
        
        if absent:
            report.append(f"\n⚠️  ABSENT ({len(absent)}):")
            for name in sorted(absent):
                report.append(f"   - {name}")
        
        report.append("="*60)
        return "\n".join(report)

inference = AttendanceInference(detector, aligner, recognizer, gallery, tracker, config)
print("✅ Inference pipeline ready!")

In [ ]:
# %% Cell 11: RUN — Enrollment
# ============================================================================
# CELL 11: RUN ENROLLMENT
# ============================================================================
# Upload your enrollment photos to /kaggle/input/enrollment/
# Expected: person1.jpg, person2.jpg, ... (one photo per person)
# ============================================================================

# print("\n🚀 Starting enrollment...")
# enrollment_results = enrollment.enroll_directory(config.ENROLLMENT_DIR)

# # Display gallery stats
# stats = gallery.get_stats()
# print(f"\n📊 Gallery Statistics:")
# print(f"   Total vectors: {stats['total_vectors']}")
# print(f"   Total persons: {stats['total_persons']}")
# print(f"   Per person:")
# for name, count in stats['persons'].items():
#     print(f"     - {name}: {count} vectors")

In [ ]:
# %% Cell 12: RUN — Video Inference
# ============================================================================
# CELL 12: RUN VIDEO INFERENCE (from Google Drive link)
# ============================================================================
# Paste your Google Drive shareable link below. The video will be downloaded
# and inference will run on it automatically.
#
# Accepted link formats:
#   - https://drive.google.com/file/d/FILE_ID/view?usp=sharing
#   - https://drive.google.com/open?id=FILE_ID
#   - Or just the FILE_ID directly
# ============================================================================

import re

# ╔══════════════════════════════════════════════════════════════╗
# ║  👇 PASTE YOUR GOOGLE DRIVE VIDEO LINK HERE 👇              ║
# ╚══════════════════════════════════════════════════════════════╝

GDRIVE_LINK = "https://drive.google.com/file/d/1dxS1TROoTp_SwYQ0iXsWU7EIfMY5BXyP/view?usp=sharing"

# ════════════════════════════════════════════════════════════════

def download_from_gdrive(link: str, output_dir: str) -> str:
    """
    Download a video from Google Drive given a shareable link.
    
    Supports formats:
    - https://drive.google.com/file/d/FILE_ID/view?usp=sharing
    - https://drive.google.com/open?id=FILE_ID  
    - Raw FILE_ID
    """
    # Install gdown if not present
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown
    
    # Extract file ID from various Google Drive URL formats
    file_id = None
    
    # Format: /file/d/FILE_ID/
    match = re.search(r'/file/d/([a-zA-Z0-9_-]+)', link)
    if match:
        file_id = match.group(1)
    
    # Format: ?id=FILE_ID or &id=FILE_ID
    if not file_id:
        match = re.search(r'[?&]id=([a-zA-Z0-9_-]+)', link)
        if match:
            file_id = match.group(1)
    
    # Assume raw file ID if no URL pattern matched
    if not file_id:
        file_id = link.strip()
    
    print(f"📥 Downloading video from Google Drive...")
    print(f"   File ID: {file_id}")
    
    output_path = os.path.join(output_dir, "gdrive_video.mp4")
    url = f"https://drive.google.com/uc?id={file_id}"
    
    gdown.download(url, output_path, quiet=False)
    
    if os.path.exists(output_path):
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"   ✅ Downloaded: {output_path} ({size_mb:.1f} MB)")
    else:
        print(f"   ❌ Download failed!")
        
    return output_path


# Download the video
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
video_path = download_from_gdrive(GDRIVE_LINK, config.OUTPUT_DIR)

# Run inference
if os.path.exists(video_path):
    # Fresh tracker for this video
    tracker_v = IdentityTracker(config)
    inference_v = AttendanceInference(
        detector, aligner, recognizer, gallery, tracker_v, config
    )
    
    print(f"\n{'='*60}")
    print(f"🎬 Processing: {os.path.basename(video_path)}")
    print(f"{'='*60}")
    
    results = inference_v.run_on_video(video_path)
    
    # Print attendance report
    report = inference_v.get_attendance_report()
    print(report)
    
    # Save attendance to JSON
    attendance_path = os.path.join(config.OUTPUT_DIR, "attendance.json")
    with open(attendance_path, "w") as f:
        json.dump({
            "video": os.path.basename(video_path),
            "attendance": inference_v.attendance,
            "gallery_stats": gallery.get_stats(),
            "inference_stats": {
                "total_frames": results.get("total_frames", 0),
                "processed_frames": results.get("processed_frames", 0),
                "elapsed_seconds": results.get("elapsed_seconds", 0),
                "fps": results.get("fps", 0),
            }
        }, f, indent=2)
    
    print(f"\n💾 Attendance saved to: {attendance_path}")
else:
    print("❌ Video file not found. Check your Google Drive link and sharing permissions.")

In [ ]:
# %% Cell 14: Benchmarking & Diagnostics
# ============================================================================
# CELL 14: STANDARD BENCHMARKING
# ============================================================================
# Measures embedding quality, gallery discrimination, and timing.
# ============================================================================

print("\n" + "="*60)
print("📏 BENCHMARKING & DIAGNOSTICS")
print("="*60)

# ─── 1. Gallery Pairwise Similarity Analysis ──────────────────────────────
print("\n1️⃣  Gallery Embedding Quality")
print("-"*40)

# Get all gallery vectors and labels
if gallery.index.ntotal > 0:
    all_vectors = faiss.rev_swig_ptr(gallery.index.get_xb(), gallery.index.ntotal * 512)
    all_vectors = np.array(all_vectors).reshape(gallery.index.ntotal, 512)
    all_labels = [name for name, _ in gallery.id_map]
    
    # Compute per-person centroid
    unique_names = sorted(set(all_labels))
    centroids = {}
    for name in unique_names:
        idxs = [i for i, l in enumerate(all_labels) if l == name]
        person_vecs = all_vectors[idxs]
        centroid = person_vecs.mean(axis=0)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-8)
        centroids[name] = centroid
        
        # Intra-class stats
        intra_sims = person_vecs @ centroid
        print(f"   {name:20s}: {len(idxs):2d} vectors | "
              f"intra-sim: {intra_sims.mean():.4f} ± {intra_sims.std():.4f} | "
              f"min: {intra_sims.min():.4f}")
    
    # Inter-class similarity matrix
    print(f"\n   📊 Inter-Class Centroid Similarity Matrix:")
    print(f"   {'':20s}", end="")
    for n in unique_names:
        print(f" {n[:8]:>8s}", end="")
    print()
    
    for n1 in unique_names:
        print(f"   {n1:20s}", end="")
        for n2 in unique_names:
            sim = float(centroids[n1] @ centroids[n2])
            marker = "██" if n1 == n2 else ("⚠️" if sim > 0.5 else "  ")
            print(f" {sim:8.4f}", end="")
        print()
    
    # Summary metrics
    inter_sims = []
    for i, n1 in enumerate(unique_names):
        for n2 in unique_names[i+1:]:
            inter_sims.append(float(centroids[n1] @ centroids[n2]))
    
    if inter_sims:
        print(f"\n   ✅ Inter-class similarity: mean={np.mean(inter_sims):.4f}, "
              f"max={np.max(inter_sims):.4f}")
        print(f"   {'👍 GOOD' if np.max(inter_sims) < 0.5 else '⚠️  WARNING: Some inter-class sims > 0.5'}")

# ─── 2. Inference Timing ─────────────────────────────────────────────────
print(f"\n2️⃣  Inference Performance")
print("-"*40)

# Time a single embedding extraction
if vit_recognizer.available:
    import torch
    dummy_face = np.random.randint(0, 255, (112, 112, 3), dtype=np.uint8)
    
    # Warmup
    for _ in range(3):
        vit_recognizer.get_embedding(dummy_face)
    
    # Benchmark
    n_runs = 20
    t0 = time.time()
    for _ in range(n_runs):
        vit_recognizer.get_embedding(dummy_face)
    vit_time = (time.time() - t0) / n_runs * 1000
    print(f"   ViT embedding:   {vit_time:.1f} ms/face ({1000/vit_time:.0f} faces/sec)")

# Time ArcFace embedding
dummy_face_arc = np.random.randint(0, 255, (112, 112, 3), dtype=np.uint8)
t0 = time.time()
for _ in range(20):
    detector._original_unpatched_get_embedding_direct(dummy_face_arc)
arc_time = (time.time() - t0) / 20 * 1000
print(f"   ArcFace embedding: {arc_time:.1f} ms/face ({1000/arc_time:.0f} faces/sec)")

# ─── 3. Detection Timing ──────────────────────────────────────────────────
dummy_frame = np.random.randint(0, 255, (480, 848, 3), dtype=np.uint8)
t0 = time.time()
for _ in range(10):
    detector.detect(dummy_frame)
det_time = (time.time() - t0) / 10 * 1000
print(f"   SCRFD detection:  {det_time:.1f} ms/frame")

# ─── 4. Rank-1 Identification Accuracy (Leave-One-Out on Gallery) ────────
print(f"\n4️⃣  Rank-1 Identification Accuracy")
print("-"*40)

if gallery.index.ntotal > 0:
    correct = 0
    total = 0
    per_person_correct = {n: 0 for n in unique_names}
    per_person_total = {n: 0 for n in unique_names}
    
    for i in range(gallery.index.ntotal):
        query = all_vectors[i:i+1].copy()
        true_label = all_labels[i]
        
        # Search k=2 (first match will be itself, second is the real nearest)
        D, I = gallery.index.search(query, k=2)
        
        # Skip self-match (index i), take the next best
        for j in range(2):
            idx = int(I[0][j])
            if idx != i:
                pred_label = all_labels[idx]
                if pred_label == true_label:
                    correct += 1
                    per_person_correct[true_label] += 1
                break
        total += 1
        per_person_total[true_label] += 1
    
    rank1_acc = correct / total * 100
    print(f"   Overall Rank-1 Accuracy: {rank1_acc:.1f}% ({correct}/{total})")
    print(f"\n   Per-person breakdown:")
    for name in unique_names:
        acc = per_person_correct[name] / per_person_total[name] * 100 if per_person_total[name] > 0 else 0
        bar = "█" * int(acc / 5) + "░" * (20 - int(acc / 5))
        print(f"   {name:20s}: {acc:5.1f}% {bar}")

# ─── 5. Threshold Sensitivity Analysis ────────────────────────────────────
print(f"\n5️⃣  Threshold Sensitivity Analysis")
print("-"*40)

if gallery.index.ntotal > 0:
    # Build genuine pairs (same person) and impostor pairs (different person)
    genuine_scores = []
    impostor_scores = []
    
    for i in range(gallery.index.ntotal):
        for j in range(i + 1, min(i + 20, gallery.index.ntotal)):  # sample nearby pairs
            sim = float(all_vectors[i] @ all_vectors[j])
            if all_labels[i] == all_labels[j]:
                genuine_scores.append(sim)
            else:
                impostor_scores.append(sim)
    
    genuine_scores = np.array(genuine_scores) if genuine_scores else np.array([0.0])
    impostor_scores = np.array(impostor_scores) if impostor_scores else np.array([0.0])
    
    print(f"   Genuine pairs:   {len(genuine_scores):4d} | mean={genuine_scores.mean():.4f} | "
          f"min={genuine_scores.min():.4f} | max={genuine_scores.max():.4f}")
    print(f"   Impostor pairs:  {len(impostor_scores):4d} | mean={impostor_scores.mean():.4f} | "
          f"min={impostor_scores.min():.4f} | max={impostor_scores.max():.4f}")
    
    # Table of TPR/FPR at different thresholds
    thresholds = [0.2, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7]
    print(f"\n   {'Threshold':>10s} {'TPR':>8s} {'FPR':>8s} {'Precision':>10s} {'F1':>8s}")
    print(f"   {'─'*10} {'─'*8} {'─'*8} {'─'*10} {'─'*8}")
    
    best_f1 = 0
    best_thresh = 0.4
    for t in thresholds:
        tp = np.sum(genuine_scores >= t)
        fn = np.sum(genuine_scores < t)
        fp = np.sum(impostor_scores >= t)
        tn = np.sum(impostor_scores < t)
        
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1 = 2 * precision * tpr / (precision + tpr) if (precision + tpr) > 0 else 0
        
        marker = " ◀── current" if abs(t - 0.4) < 0.01 else ""
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
        
        print(f"   {t:10.2f} {tpr:8.4f} {fpr:8.4f} {precision:10.4f} {f1:8.4f}{marker}")
    
    print(f"\n   🏆 Best threshold: {best_thresh:.2f} (F1={best_f1:.4f})")

# ─── 6. ROC Curve & AUC ─────────────────────────────────────────────────
print(f"\n6️⃣  ROC Curve & AUC")
print("-"*40)

if gallery.index.ntotal > 0 and len(genuine_scores) > 0 and len(impostor_scores) > 0:
    # Compute ROC points
    all_scores = np.concatenate([genuine_scores, impostor_scores])
    all_truth = np.concatenate([np.ones(len(genuine_scores)), np.zeros(len(impostor_scores))])
    
    # Sort thresholds
    roc_thresholds = np.linspace(all_scores.min() - 0.01, all_scores.max() + 0.01, 200)
    tpr_list = []
    fpr_list = []
    
    for t in roc_thresholds:
        tp = np.sum(genuine_scores >= t)
        fn = np.sum(genuine_scores < t)
        fp = np.sum(impostor_scores >= t)
        tn = np.sum(impostor_scores < t)
        
        tpr_list.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    
    tpr_arr = np.array(tpr_list)
    fpr_arr = np.array(fpr_list)
    
    # AUC via trapezoidal rule (sort by FPR ascending)
    sorted_idx = np.argsort(fpr_arr)
    fpr_sorted = fpr_arr[sorted_idx]
    tpr_sorted = tpr_arr[sorted_idx]
    auc = np.trapz(tpr_sorted, fpr_sorted)
    
    print(f"   AUC (Area Under ROC Curve): {auc:.4f}")
    print(f"   {'🏆 Excellent' if auc > 0.99 else '✅ Good' if auc > 0.95 else '⚠️ Needs improvement'}")
    
    # Key operating points
    print(f"\n   Key operating points:")
    for target_fpr in [0.001, 0.01, 0.1]:
        idx = np.argmin(np.abs(fpr_arr - target_fpr))
        print(f"   FPR={target_fpr:.3f} → TPR={tpr_arr[idx]:.4f} (threshold={roc_thresholds[idx]:.3f})")
    
    # EER (Equal Error Rate)
    fnr_arr = 1 - tpr_arr
    eer_idx = np.argmin(np.abs(fpr_arr - fnr_arr))
    eer = (fpr_arr[eer_idx] + fnr_arr[eer_idx]) / 2
    print(f"\n   EER (Equal Error Rate): {eer:.4f} ({eer*100:.2f}%)")
    print(f"   {'🏆 Excellent' if eer < 0.01 else '✅ Good' if eer < 0.05 else '⚠️ Needs improvement'}")

# ─── 7. Silhouette Score (Classification R² Equivalent) ──────────────────
print(f"\n7️⃣  Silhouette Score (Clustering Quality — R² Equivalent)")
print("-"*40)

if gallery.index.ntotal > 0 and len(unique_names) > 1:
    # Silhouette: for each point, compute (b - a) / max(a, b)
    #   a = mean intra-cluster distance
    #   b = mean nearest-cluster distance
    # Range: [-1, 1]. Higher = better separation.
    
    silhouette_scores = []
    for i in range(gallery.index.ntotal):
        label_i = all_labels[i]
        vec_i = all_vectors[i]
        
        # Intra-cluster: mean cosine distance to same-class vectors
        same_idxs = [j for j in range(gallery.index.ntotal) if all_labels[j] == label_i and j != i]
        if not same_idxs:
            continue
        same_vecs = all_vectors[same_idxs]
        a_i = 1.0 - float(np.mean(vec_i @ same_vecs.T))  # cosine distance
        
        # Inter-cluster: mean distance to nearest other cluster
        b_i = float('inf')
        for other_name in unique_names:
            if other_name == label_i:
                continue
            other_idxs = [j for j in range(gallery.index.ntotal) if all_labels[j] == other_name]
            other_vecs = all_vectors[other_idxs]
            mean_dist = 1.0 - float(np.mean(vec_i @ other_vecs.T))
            b_i = min(b_i, mean_dist)
        
        s_i = (b_i - a_i) / max(a_i, b_i) if max(a_i, b_i) > 0 else 0
        silhouette_scores.append(s_i)
    
    mean_silhouette = np.mean(silhouette_scores)
    print(f"   Mean Silhouette Score: {mean_silhouette:.4f}")
    print(f"   Interpretation:")
    print(f"     1.0  = Perfect separation (every embedding perfectly clustered)")
    print(f"     0.5+ = Good separation (clear identity clusters)")
    print(f"     0.0  = Overlapping clusters (confusion likely)")
    print(f"    -1.0  = Wrong clustering (embeddings assigned to wrong identity)")
    quality = ("🏆 Excellent" if mean_silhouette > 0.7 else 
               "✅ Good" if mean_silhouette > 0.5 else 
               "⚠️ Moderate" if mean_silhouette > 0.25 else "❌ Poor")
    print(f"   Result: {quality}")
    
    # Note about R²
    print(f"\n   📝 Note on R² Score:")
    print(f"   R² is a regression metric (predicting continuous values).")
    print(f"   For classification/recognition, these are the standard equivalents:")
    print(f"     • Silhouette Score → measures cluster quality (like R² for clusters)")
    print(f"     • AUC-ROC         → measures discriminative ability")
    print(f"     • EER             → single-number error rate (lower = better)")
    print(f"     • Rank-1 Accuracy → % of correct top-1 identification")

# ─── 8. Pipeline Summary ────────────────────────────────────────────────
print(f"\n8️⃣  Pipeline Summary")
print("-"*40)
print(f"   Gallery size:     {gallery.index.ntotal} vectors")
print(f"   Gallery persons:  {len(set(all_labels)) if gallery.index.ntotal > 0 else 0}")
print(f"   Recognition model: {'AdaFace ViT-Base KP-RPE' if vit_recognizer.available else 'ArcFace ResNet'}")
print(f"   Detection model:  SCRFD-10G")
print(f"   Tracker:          ByteTrack")
print(f"   Index type:       FAISS FlatIP (exact search)")

print("\n" + "="*60)
print("📏 BENCHMARKING COMPLETE")
print("="*60)